In [2]:
import pandas as pd
df = pd.read_csv('./data/cytof.csv', index_col=0)

In [9]:
df_ts = df[(df.observableId == 'p.ERK') & (df.simulationConditionId.str.endswith('__EGF'))].pivot_table(index='preequilibrationConditionId', columns='time', values='measurement')

In [10]:
df_ts

time,0.0,5.5,7.0,9.0,12.0,13.0,14.0,17.0,23.0,30.0,35.0,40.0,60.0
preequilibrationConditionId,,,,,,,,,,,,,
c184A1,2.610448,3.562203,3.503852,3.677882,NaN,3.522100,NaN,3.333949,3.631606,3.225276,NaN,2.997180,2.794183
c184B5,2.303940,3.657244,3.324000,3.025286,NaN,3.918409,NaN,3.321868,3.713584,2.687524,NaN,2.625606,2.612064
cBT20,3.805443,3.990476,4.411401,4.240665,NaN,3.993469,NaN,4.529850,3.900909,4.510031,NaN,3.737119,4.148405
cBT474,3.089931,3.320202,3.088775,3.261580,NaN,3.459100,NaN,3.215458,2.978081,3.171048,NaN,2.903087,3.052457
cBT483,2.941606,3.898188,3.991941,3.743703,NaN,3.468802,NaN,3.100587,3.201544,2.630544,NaN,2.680406,2.462154
cBT549,2.774730,3.805003,3.986812,3.723047,NaN,3.315204,NaN,3.625836,3.330300,3.653468,NaN,3.253072,3.417519
cCAL148,1.969755,3.532953,3.275293,3.478243,NaN,3.335307,NaN,3.302267,3.362060,3.269086,NaN,2.330384,2.884917
cCAL51,2.625466,4.037332,3.771651,3.893120,NaN,3.844919,NaN,3.386502,3.715885,3.631095,NaN,3.872647,3.624477
cCAL851,3.290168,3.821046,3.551637,3.818823,NaN,3.651139,NaN,3.306332,3.510824,3.157344,NaN,3.232957,2.764083


In [21]:
import numpy as np
from dataclasses import dataclass
from scipy.optimize import curve_fit

def _sigmoid(x, center, width):
    # width > 0 makes the transition slope ~1/width; allow negative for decreasing steps
    return 1.0 / (1.0 + np.exp(-(x - center) / width))

def double_sigmoid_product(x, A, c1, w1, c2, w2, B):
    """
    Model: y = B + A * sigmoid((x-c1)/w1) * sigmoid((x-c2)/w2)
      A: amplitude (scale of the product)
      B: baseline (offset)
      c1, c2: centers of the two sigmoids
      w1, w2: widths (slope controls; can be negative for decreasing steps)
    """
    return B + A * _sigmoid(x, c1, w1) * _sigmoid(x, c2, w2)

@dataclass
class FitResult:
    params: dict
    popt: np.ndarray
    pcov: np.ndarray
    stderr: np.ndarray

def fit_double_sigmoid(x, y, max_nfev=20000):
    """
    Fit y ~ B + A * s1 * s2 with scipy.optimize.curve_fit.

    Parameters
    ----------
    x, y : 1D arrays
    y_sigma : optional 1D array of observation std devs (weights = 1/sigma^2)
    p0 : optional initial guess [A, c1, w1, c2, w2, B]
    bounds : optional ((lower,...), (upper,...)) for parameters
    robust : if True, do two-stage fit with Huber-like reweighting
    max_nfev : max function evaluations

    Returns
    -------
    FitResult with dict of params, covariance, and standard errors.
    """
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()

    # Default initial guesses
    B0 = np.nanpercentile(y, 5)
    top = np.nanpercentile(y, 95)
    A0 = max(top - B0, 1e-6)
    # centers near quartiles; widths span ~1/10 of x-range
    xlo, xhi = np.nanmin(x), np.nanmax(x)
    span = max(xhi - xlo, 1e-9)
    c10 = xlo + 0.3 * span
    c20 = xlo + 0.7 * span
    w10 = 0.1 * span
    w20 = 0.1 * span
    p0 = [A0, c10, w10, c20, w20, B0]

    xlo, xhi = np.nanmin(x), np.nanmax(x)
    span = max(xhi - xlo, 1e-9)
    # Allow A to be either sign; widths avoid 0 to keep gradients stable
    lower = [-np.inf, xlo - 2*span, -10*span, xlo - 2*span, -10*span, -np.inf]
    upper = [ np.inf, xhi + 2*span,  10*span, xhi + 2*span,  10*span,  np.inf]
    bounds = (lower, upper)

    def _fit():
        popt, pcov = curve_fit(
            double_sigmoid_product,
            x,
            y,
            p0=p0,
            bounds=bounds,
            max_nfev=max_nfev,
        )
        return popt, pcov

    # Stage 1: standard (or weighted) LS
    popt, pcov = _fit()

    # Standard errors from covariance
    stderr = np.sqrt(np.maximum(np.diag(pcov), 0.0))

    params = {
        "A": popt[0],
        "c1": popt[1],
        "w1": popt[2],
        "c2": popt[3],
        "w2": popt[4],
        "B": popt[5],
        "A_se": stderr[0],
        "c1_se": stderr[1],
        "w1_se": stderr[2],
        "c2_se": stderr[3],
        "w2_se": stderr[4],
        "B_se": stderr[5],
    }
    return FitResult(params=params, popt=popt, pcov=pcov, stderr=stderr)

In [22]:
df_ts.iloc[0,:].values

y = df_ts.iloc[0,:].values
x = df_ts.columns.values.astype(float)
x = x[~np.isnan(y)]
y = y[~np.isnan(y)]

In [23]:

res = fit_double_sigmoid(x, y)
print("Fitted params:")
for k in ["A", "c1", "w1", "c2", "w2", "B"]:
    print(f"  {k:>2} = {res.params[k]:.4f} ± {res.params[k+'_se']:.4f}")

Fitted params:
   A = -0.7455 ± 0.5792
  c1 = 37.0718 ± 50.4204
  w1 = 15.0714 ± 34.2442
  c2 = 29.8739 ± 1.7318
  w2 = 0.2368 ± 0.9220
   B = 3.4060 ± 0.1701
